In [3]:
import numpy
from hmmlearn import hmm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

#regimes (trending, mean reverting, high volatility, low volatility)

In [25]:
import yfinance as yf

def clean_yfinance_data(ticker, start, end):
    """Baixa dados e remove MultiIndex se existir"""
    data = yf.download(ticker, start=start, end=end)
    
    # Se tiver MultiIndex nas colunas, remove
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)
    
    return data['Close']

# Usar a função
sp500 = clean_yfinance_data("^GSPC", "2000-01-01", "2025-12-31")
ibov = clean_yfinance_data("^BVSP", "2000-01-01", "2025-12-31")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [26]:
def prepare_features(prices, window=90):
    """prepara feature para os 4 estados"""
    feature = pd.DataFrame(prices)

    feature['returns'] = np.log(prices / prices.shift(1))

    feature['volatility'] = feature['returns'].rolling(window).std()

    feature['momentum'] = (prices / prices.rolling(window).mean()) - 1

    feature['mean_reversion'] = -feature['returns'].rolling(window).mean()

    feature['vol_regime'] = feature['volatility'] / feature['volatility'].rolling(252).mean()

    return feature.dropna()


In [24]:
df = pd.read_csv('../db/markets.csv', parse_dates=True)
df.set_index('Date', inplace=True)
df

,ibovespa_br,s&p500_eua,dax_alemanha,shanghai_china,nikkei_japao,merv_argentina,ftse_uk
Date,,,,,,,
2000-01-03,16930.0,1455.219971,6750.759766,NaN,NaN,552.0,NaN
2000-01-04,15851.0,1399.420044,6586.950195,1406.370972,19002.859375,523.0,6665.899902
2000-01-05,16245.0,1402.109985,6502.069824,1409.682007,18542.550781,533.0,6535.899902
2000-01-06,16107.0,1403.449951,6474.919922,1463.942017,18168.269531,529.0,6447.200195
2000-01-07,16309.0,1441.469971,6780.959961,1516.604004,18193.410156,522.0,6504.799805
...,...,...,...,...,...,...,...
2026-03-24,182509.0,6556.370117,22636.910156,3881.280029,52252.281250,NaN,9965.200195
2026-03-25,185424.0,6591.899902,22957.080078,3931.836914,53749.621094,2805317.0,10106.799805
2026-03-26,182733.0,6477.160156,22612.970703,3889.083008,53603.648438,2769369.0,9972.200195


In [ ]:
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm


def identifying_regimes(features_df, n_states=4):

    X = features_df.values

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)

    model = hmm.GaussianHMM(
        n_components=n_states,
        covariance_type='full',
        n_iter=1000,
        random_state=42
    )

    model.fit(X_scaled)

    regimes = model.predict(X_scaled)

    features_df['regime'] = regimes

    return features_df

In [31]:
regimes_markets = {}

for market in df.columns:

    try:

        prices = df[market].dropna()

        features = prepare_features(prices)

        regimes = identifying_regimes(features)

        regimes_markets[market] = regimes

        print(f'{market} OK')

    except Exception as e:

        print(f'Erro em {market}: {e}')



ibovespa_br OK
s&p500_eua OK
dax_alemanha OK
shanghai_china OK
nikkei_japao OK
merv_argentina OK
ftse_uk OK
